In [1]:
pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.6 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


### charger le model Qwen2.5-Math-7B-Instruct

In [5]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
model_name = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

### Evaluation for the entire dataset


In [6]:
"""
Benchmark de vérification d'hypothèses de théorèmes mathématiques
avec Qwen2.5-7B-Instruct (4-bit) sur Kaggle.

Le fichier JSON doit contenir une liste d'objets du type :

{
    "theorem_id": "T21",
    "name": "Adjacent Sequences Theorem",
    "copy": [...],
    "expected": [...],
    "is_correct": false,
    "reason": "...",
    "validation_type": "missing_hypothesis"
}

`validation_type` décrit la manière dont la liste d'hypothèses doit être
validée, et non simplement un "type d'erreur".

Valeurs possibles :
- "correct"
- "missing_hypothesis"
- "multiple_missing"
- "invented_hypothesis"
- "valid_implication"

`valid_implication` est considéré comme mathématiquement correct :
une hypothèse plus forte peut remplacer une hypothèse requise si elle
l'implique logiquement.
"""

import json
import re
import time
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# Adapter selon ton dataset Kaggle

INPUT_JSON_PATH = "/kaggle/input/datasets/mathurasanthalingam/benchmark/benchmark_data.json"
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_results.csv"

# ============================================================
# CHARGEMENT DU MODELE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(model_name)


# ============================================================
# PROMPTS
# ============================================================

SYSTEM_PROMPT = """You are a mathematics expert tasked with evaluating whether a given list of hypotheses for a theorem is correct.

The benchmark evaluates whether the proposed hypotheses correctly cover the hypotheses required by the standard version of the theorem.

IMPORTANT CONCEPTUAL RULE:
The benchmark distinguishes between:
- a hypothesis being genuinely missing;
- a hypothesis being replaced by an incorrect or unrelated hypothesis;
- a hypothesis being replaced by a stronger hypothesis that validly implies the required one.

A replacement is considered a replacement, NOT simultaneously an omission plus an addition.

EVALUATION RULES:

1. Identify the standard required hypotheses of the theorem.

2. Compare the proposed list of hypotheses with the required hypotheses.

3. A required hypothesis may be satisfied directly by the corresponding hypothesis.

4. A required hypothesis may also be satisfied by a DIFFERENT, STRONGER hypothesis if that stronger hypothesis logically implies the required one.

   Example:
   If the theorem requires:
   "f is continuous on [a,b]"

   and the proposed list contains:
   "f is differentiable on ]a,b["

   this is NOT automatically a valid replacement unless the domains and
   precise mathematical statements make the implication valid.

   When a proposed hypothesis genuinely implies a required hypothesis,
   it may REPLACE that required hypothesis.

5. IMPORTANT — valid replacement versus redundant addition:

   If a stronger hypothesis replaces a required hypothesis, it is accepted.

   Example:
   Required:
   "f is continuous"

   Proposed:
   "f is differentiable"

   If differentiability logically implies the required continuity in the
   relevant setting, the proposed list can be considered correct.

   However, if the proposed list contains BOTH the stronger hypothesis AND
   the weaker hypothesis it already implies, the weaker one is redundant.
   An unnecessary additional hypothesis must be classified as
   "invented_hypothesis".

6. A hypothesis that replaces an expected hypothesis but does NOT satisfy
   that requirement is classified as "invented_hypothesis".

   This is a deliberate benchmark rule.

   Example:
   Required:
   - E is a Hilbert space
   - F is a closed subspace of E

   Proposed:
   - F is a subspace of E (not necessarily closed)

   This is treated as a REPLACEMENT by an incorrect hypothesis and therefore
   classified as "invented_hypothesis".

   Do NOT decompose this into:
   - missing "E is a Hilbert space"
   - plus invented "F is not necessarily closed"

   The benchmark considers the student's statement to be a replacement of
   an expected hypothesis by an incorrect one.

7. Only take into account the hypotheses explicitly listed.
   Do not assume implicit hypotheses that are not mentioned.

8. A mathematically true statement is not necessarily a valid hypothesis
   for the theorem. It must actually be required or validly replace a
   required hypothesis.

9. Do not discuss whether the theorem itself is mathematically true.
   Evaluate only whether the proposed hypothesis list correctly matches
   the hypotheses needed for the standard theorem.

DECISION PROCEDURE (apply in this order):

a) Identify the hypotheses required by the standard version of the theorem.

b) For each proposed hypothesis, determine whether it:
   - directly matches one required hypothesis;
   - VALIDLY REPLACES one required hypothesis because it is strictly stronger and logically implies it;
   - does not satisfy any required hypothesis.

c) A replacement is valid ONLY if the implication is mathematically valid on its own, without requiring an unstated additional assumption.

d) If a proposed hypothesis is different from a required hypothesis, do NOT assume that it is stronger merely because it is related, similar, compatible, or intuitively more restrictive.

e) If a proposed hypothesis does NOT logically imply the required hypothesis, it does NOT satisfy that requirement. It must instead be treated as an invented_hypothesis.

f) If a proposed hypothesis replaces a required hypothesis with an incorrect or unrelated condition, this is ONE replacement error and must be classified as invented_hypothesis. Do NOT reinterpret it as both a missing hypothesis and an invented hypothesis.

g) If one or more required hypotheses have no corresponding hypothesis in the proposed list, classify the example as missing_hypothesis or multiple_missing.

h) If exactly one required hypothesis is missing, use missing_hypothesis.

i) If more than one required hypothesis is missing, use multiple_missing.

j) Only if every required hypothesis is covered exactly once, either directly or through a mathematically valid stronger replacement, AND no unrelated/redundant hypothesis is present, is the list correct.

IMPORTANT PRIORITY RULE:
If a proposed hypothesis REPLACES a required hypothesis but the replacement is invalid, classify the error as invented_hypothesis rather than simultaneously reporting a missing_hypothesis.

Example:
Required:
- E is a Hilbert space
- F is a closed subspace of E

Proposed:
- F is a subspace of E (not necessarily closed)

This is a REPLACEMENT of the required condition on F, not merely an omission plus an addition.
Because "F is a subspace of E" does NOT imply "F is a closed subspace of E", the correct classification is invented_hypothesis.

IMPORTANT:
Do not classify a hypothesis as a valid stronger replacement unless the logical implication is genuinely valid.

For example:
- "f is differentiable" can imply "f is continuous" under the relevant standard mathematical setting.
- "f is continuous on ]a,b[" does NOT imply "f is continuous on [a,b]".
- "F is a subspace" does NOT imply "F is a closed subspace".
- "integrable" does NOT imply "continuous".
VALIDATION TYPES:

- "correct":
  All required hypotheses are present directly, with no incorrect or
  unnecessary additional hypothesis.

- "valid_implication":
  All required hypotheses are covered, but at least one is satisfied through
  a stronger hypothesis that validly implies it.
  This is also considered mathematically correct.

- "missing_hypothesis":
  Exactly one required hypothesis is absent, without an incorrect
  replacement accounting for that requirement.

- "multiple_missing":
  More than one required hypothesis is absent.

- "invented_hypothesis":
  A hypothesis is incorrectly substituted for an expected hypothesis,
  is unrelated to the theorem, is unnecessarily added, or is otherwise
  not a valid hypothesis for the theorem.

OUTPUT FORMAT (strict JSON, nothing else, no markdown, no explanation outside the JSON):

{
    "label": "correct" or "incorrect",
    "validation_type": null or one of [
        "correct",
        "valid_implication",
        "missing_hypothesis",
        "multiple_missing",
        "invented_hypothesis"
    ],
    "reason": "concise explanation of the validation decision, identifying the relevant hypothesis or logical implication when appropriate"
}

IMPORTANT:
- If validation_type is "correct" or "valid_implication", label MUST be "correct".
- If validation_type is "missing_hypothesis", "multiple_missing", or
  "invented_hypothesis", label MUST be "incorrect".
- Do NOT use "valid_implication" for an incorrect implication.
- Do NOT classify a replacement as both missing and invented.
- A valid stronger replacement is accepted.
- An incorrect replacement is classified as "invented_hypothesis".

CRITICAL INSTRUCTION:
You MUST respond with ONLY the JSON object.
Do NOT include any explanation, introduction, or text before or after the JSON.
Start your response with "{" and end your response with "}".
Any text outside the JSON will cause a parsing error.
"""


USER_PROMPT_TEMPLATE = """Theorem: {theorem_name}

Proposed hypotheses:
{hypotheses_list}

Evaluate whether this list correctly covers the hypotheses required by the standard version of the theorem.

Remember:
- A stronger hypothesis may REPLACE a required hypothesis if it logically implies it.
- A valid stronger replacement is classified as "valid_implication" and is CORRECT.
- An incorrect replacement is classified as "invented_hypothesis".
- A replacement must NOT be decomposed into a missing hypothesis plus an invented hypothesis.
- An unnecessary additional hypothesis is "invented_hypothesis".

CRITICAL: Return ONLY the JSON object, starting with "{{" and ending with "}}". No explanation.
"""


def build_user_prompt(theorem_name, hypotheses):
    hyp = "\n".join(f"- {h}" for h in hypotheses)

    return USER_PROMPT_TEMPLATE.format(
        theorem_name=theorem_name,
        hypotheses_list=hyp,
    )


# ============================================================
# APPEL LLM
# ============================================================

def call_llm(theorem_name, hypotheses):

    user_prompt = build_user_prompt(
        theorem_name,
        hypotheses,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer.strip()


# ============================================================
# PARSER JSON
# ============================================================

def parse_llm_output(raw_text):

    if raw_text is None:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_response",
        }

    cleaned = raw_text.strip()

    # --------------------------------------------------------
    # 1. Nettoyage éventuel des balises Markdown
    # --------------------------------------------------------

    cleaned = re.sub(
        r"```(?:json)?",
        "",
        cleaned,
        flags=re.IGNORECASE,
    ).strip()

    cleaned = cleaned.replace("```", "").strip()

    # --------------------------------------------------------
    # 2. Chercher des objets JSON candidats
    #
    # On évite le greedy match "{.*}" qui pouvait englober
    # une grande quantité de texte non JSON.
    # --------------------------------------------------------

    decoder = json.JSONDecoder()

    candidates = []

    for match in re.finditer(r"\{", cleaned):

        start = match.start()

        try:
            obj, end = decoder.raw_decode(cleaned[start:])
            candidates.append(obj)
        except json.JSONDecodeError:
            continue

    # --------------------------------------------------------
    # 3. Aucun JSON trouvé
    # --------------------------------------------------------

    if not candidates:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_valid_json_object_found",
        }

    # --------------------------------------------------------
    # 4. Chercher de préférence un objet ayant la bonne structure
    # --------------------------------------------------------

    parsed = None

    for candidate in candidates:

        if not isinstance(candidate, dict):
            continue

        if (
            "label" in candidate
            and "validation_type" in candidate
            and "reason" in candidate
        ):
            parsed = candidate
            break

    # Si aucun objet ne possède exactement les champs attendus,
    # on prend malgré tout le premier dictionnaire trouvé.
    parsed = None
    for candidate in candidates:
        if not isinstance(candidate, dict):
            continue
        if "label" in candidate and "validation_type" in candidate and "reason" in candidate:
            parsed = candidate
            break
    
    if parsed is None:
        return {
            "label": None,
            "validation_type": None,
            "reason": None,
            "parse_error": "no_object_with_required_keys",
        }

    # --------------------------------------------------------
    # 5. Validation minimale du contenu
    # --------------------------------------------------------

    label = parsed.get("label")
    validation_type = parsed.get("validation_type")
    reason = parsed.get("reason")

    valid_labels = {"correct", "incorrect"}

    valid_validation_types = {
        "exact_match",
        "stronger_hypothesis",
        "missing_hypothesis",
        "multiple_missing",
        "invented_hypothesis",
        "valid_implication",
        None,
    }

    if label not in valid_labels:
        return {
            "label": label,
            "validation_type": validation_type,
            "reason": reason,
            "parse_error": f"invalid_label: {label}",
        }

    if validation_type not in valid_validation_types:
        return {
            "label": label,
            "validation_type": validation_type,
            "reason": reason,
            "parse_error": (
                f"invalid_validation_type: {validation_type}"
            ),
        }

    return {
        "label": label,
        "validation_type": validation_type,
        "reason": reason,
        "parse_error": None,
    }


# ============================================================
# CHARGEMENT DATASET
# ============================================================

print("Loading dataset...")

with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"{len(dataset)} examples loaded.")


# ============================================================
# BENCHMARK
# ============================================================

results = []

start = time.time()

for i, sample in enumerate(dataset):

    theorem_name = sample["name"]
    hypotheses = sample["copy"]

    try:

        raw = call_llm(
            theorem_name,
            hypotheses,
        )

        parsed = parse_llm_output(raw)

    except Exception as e:

        raw = None

        parsed = {
            "label": None,

            # [MODIFICATION]
            "validation_type": None,

            "reason": None,
            "parse_error": str(e),
        }

    results.append({

        "theorem_id": sample["theorem_id"],

        "name": theorem_name,

        "hypotheses": sample["copy"],

        "expected": sample["expected"],

        "ground_truth_label":
            "correct" if sample["is_correct"] else "incorrect",

        # [MODIFICATION]
        # On récupère désormais le type de validation produit par le
        # générateur du benchmark.
        "ground_truth_validation":
            sample["validation_type"],

        "ground_truth_reason":
            sample["reason"],

        "prediction_label":
            parsed["label"],

        # [MODIFICATION]
        "prediction_validation":
            parsed["validation_type"],

        "prediction_reason":
            parsed["reason"],

        "parse_error":
            parsed["parse_error"],

        "raw_response":
            raw,
    })

    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(dataset)} examples processed")


# ============================================================
# SAUVEGARDE
# ============================================================

df = pd.DataFrame(results)

df.to_csv(
    OUTPUT_CSV_PATH,
    index=False,
)

print()
print("CSV saved to:")
print(OUTPUT_CSV_PATH)


# ============================================================
# METRIQUES
# ============================================================

valid = df[df["parse_error"].isna()].copy()

label_accuracy = (
    valid["ground_truth_label"]
    ==
    valid["prediction_label"]
).mean()

# [MODIFICATION]
# On compare désormais les types de validation plutôt que les anciens
# "error_type".
validation_accuracy = (
    valid["ground_truth_validation"].fillna("none")
    ==
    valid["prediction_validation"].fillna("none")
).mean()

# [MODIFICATION]
# Métrique supplémentaire : parmi les exemples mathématiquement corrects,
# le LLM distingue-t-il correctement "correct" de "valid_implication" ?
correct_validation_accuracy = (
    valid["ground_truth_validation"].isin(
        ["correct", "valid_implication"]
    )
    ==
    valid["prediction_validation"].isin(
        ["correct", "valid_implication"]
    )
).mean()


print()
print("=" * 60)
print("RESULTS")
print("=" * 60)

print(f"Examples                    : {len(df)}")
print(f"Valid generations           : {len(valid)}")
print(f"Label accuracy              : {label_accuracy:.4f}")
print(f"Validation type accuracy    : {validation_accuracy:.4f}")
print(
    f"Correct/incorrect accuracy  : "
    f"{correct_validation_accuracy:.4f}"
)
print(f"Execution time              : {(time.time()-start)/60:.2f} min")

display(df.head())

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading dataset...
100 examples loaded.
10/100 examples processed
20/100 examples processed
30/100 examples processed
40/100 examples processed
50/100 examples processed
60/100 examples processed
70/100 examples processed
80/100 examples processed
90/100 examples processed
100/100 examples processed

CSV saved to:
/kaggle/working/benchmark_results.csv

RESULTS
Examples                    : 100
Valid generations           : 95
Label accuracy              : 0.4526
Validation type accuracy    : 0.0526
Correct/incorrect accuracy  : 0.0737
Execution time              : 18.29 min


,theorem_id,name,hypotheses,expected,ground_truth_label,ground_truth_validation,ground_truth_reason,prediction_label,prediction_validation,prediction_reason,parse_error,raw_response
0,T21,Adjacent Sequences Theorem,"[(uₙ) is increasing and (vₙ) is decreasing, vₙ...","[(uₙ) is increasing and (vₙ) is decreasing, vₙ...",correct,exact_match,hypotheses match exactly.,correct,valid_implication,Both 'uₙ' is increasing and 'vₙ' is decreasing...,None,"{""label"": ""correct"", ""validation_type"": ""valid..."
1,T01,Rolle's Theorem,"[f is differentiable on ]a, b[]","[f is continuous on [a, b], f is differentiabl...",incorrect,multiple_missing,"missing: f is continuous on [a, b]; missing: f...",correct,valid_implication,The proposed hypothesis 'f is differentiable o...,None,"{""label"": ""correct"", ""validation_type"": ""valid..."
2,T24,Cauchy-Lipschitz Theorem,"[f is continuous in t, f is Lipschitz in y, an...","[f is continuous in t, f is Lipschitz in y, an...",correct,exact_match,hypotheses match exactly.,correct,valid_implication,Both 'f is continuous in t' and 'f is Lipschit...,None,"{""label"": ""correct"", ""validation_type"": ""valid..."
3,T22,Cauchy Criterion (sequences),[(uₙ) is a sequence in a complete space],[(uₙ) is a sequence in a complete space],correct,exact_match,hypotheses match exactly.,correct,valid_implication,The completeness of the space implies the Cauc...,None,"{""label"": ""correct"", ""validation_type"": ""valid..."
4,T03,Intermediate Value Theorem,"[f is of class C¹ on I, y is between f(a) and ...","[f is continuous on [a, b], y is between f(a) ...",correct,stronger_hypothesis,stronger_hypothesis: f is of class C¹ on I imp...,correct,valid_implication,f is of class C¹ on I implies f is continuous ...,None,"{""label"": ""correct"", ""validation_type"": ""valid..."


In [7]:
valid = df[df["parse_error"].isna()].copy()

# 1. Le modèle est-il cohérent avec lui-même (label vs validation_type) ?
valid["self_consistent"] = valid.apply(
    lambda r: (r["prediction_label"] == "correct") == (r["prediction_validation"] in ["correct", "valid_implication"]),
    axis=1
)
print("Auto-cohérence du modèle:", valid["self_consistent"].mean())
print(valid[~valid["self_consistent"]][["prediction_label", "prediction_validation"]].head(10))

# 2. Le vocabulaire des réponses correspond-il exactement à ce qui est attendu ?
print("\nValeurs uniques produites par le modèle pour validation_type:")
print(valid["prediction_validation"].value_counts(dropna=False))

# 3. Matrice de confusion complète
print("\nCrosstab ground_truth vs prediction:")
print(pd.crosstab(valid["ground_truth_validation"].fillna("none"),
                   valid["prediction_validation"].fillna("none")))

Auto-cohérence du modèle: 1.0
Empty DataFrame
Columns: [prediction_label, prediction_validation]
Index: []

Valeurs uniques produites par le modèle pour validation_type:
prediction_validation
valid_implication      88
invented_hypothesis     7
Name: count, dtype: int64

Crosstab ground_truth vs prediction:
prediction_validation    invented_hypothesis  valid_implication
ground_truth_validation                                        
exact_match                                0                 27
invented_hypothesis                        5                 24
missing_hypothesis                         1                 21
multiple_missing                           1                  7
stronger_hypothesis                        0                  9


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/mathurasanthalingam/benchmark/benchmark_data.json


In [2]:
import os
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/.virtual_documents/__notebook_source__.ipynb


### Si y'a des lignes qui n'ont pas pu etre parsée

In [ ]:
import pandas as pd
import time

# ============================================================
# PROMPTS
# ============================================================
SYSTEM_PROMPT = """You are a mathematics expert tasked with evaluating whether a given list of hypotheses for a theorem is correct — i.e., necessary and sufficient for the theorem to be applicable.

EVALUATION RULES:
1. A theorem has a set of hypotheses that are truly NECESSARY AND SUFFICIENT for it to be applicable (including, where relevant, problem data such as an initial condition, a domain of definition, a reference point, etc.).
2. Compare the proposed list of hypotheses to this actual required set.
3. A proposed hypothesis that is STRONGER than the actually required one is ACCEPTED, provided it logically implies it AND it REPLACES the required hypothesis (rather than being added alongside it).
   Example: if the theorem requires "f continuous" and the list REPLACES this with "f differentiable", this is CORRECT because differentiable ⇒ continuous, and no separate "f continuous" hypothesis is also present.
4. IMPORTANT — Redundant additions are NOT acceptable, even if logically valid: if the list already contains a hypothesis sufficient to satisfy a requirement, and an ADDITIONAL hypothesis is included that is merely implied by (redundant with) one already in the list, this extra hypothesis must be flagged as an invented/unnecessary hypothesis. Being mathematically true or logically implied does NOT make a hypothesis necessary in the list.
   Example: if the list contains "f is Lipschitz in y" (already sufficient) AND ALSO "f is continuous in y" (implied by the first), the second is redundant and must be flagged as "invented_hypothesis" — do NOT mark the list as correct just because the added hypothesis happens to be true.
5. Only take into account the hypotheses listed. Do not assume implicit hypotheses that are not mentioned as being "understood".
6. Do not discuss the mathematical truth of the theorem itself, only the completeness and correctness of the provided hypothesis list relative to the standard version of the theorem.

DECISION PROCEDURE (apply in this order):
a) List the actual necessary-and-sufficient hypotheses for the standard version of the theorem.
b) Check if every proposed hypothesis maps to a necessary one, either directly or as a valid REPLACEMENT (stronger version implying it, per rule 3).
c) Check if any proposed hypothesis is NOT needed to satisfy any requirement — i.e., it is redundant with another hypothesis already in the list, or entirely unrelated to the theorem. If so → "invented_hypothesis", regardless of whether it is mathematically true.
d) Check if any necessary hypothesis has no corresponding entry in the list at all → missing.
e) Only if every necessary hypothesis is covered exactly once (directly or via valid replacement) AND no extra/redundant hypothesis is present → label = "correct".

If the hypotheses are necessary and sufficient, the label is "correct" and error_type must be null.

If the hypotheses are NOT correct, the label is "incorrect" and error_type must be exactly one of the following categories (choose the single most fitting one):
- "missing_hypothesis": exactly one necessary hypothesis is missing.
- "multiple_missing": more than one necessary hypothesis is missing.
- "invented_hypothesis": a hypothesis that is not necessary for the theorem, it can be a hypothesis is present but stated imprecisely or incorrectly (e.g., wrong quantifier, wrong domain, wrong scope) so that it does not actually match what is required, or a  an extra hypothesis is added that is not necessary for the theorem. 


OUTPUT FORMAT (strict JSON, nothing else, no markdown, no explanation outside the JSON):
{
  "label": "correct" or "incorrect",
  "error_type": null or one of ["missing_hypothesis", "multiple_missing", "invented_hypothesis"],
  "reason": "concise explanation of why the hypotheses are correct/sufficient, or which specific hypothesis is missing/wrong/extra and why, including relevant logical implications"
}
"""

USER_PROMPT_TEMPLATE = """Theorem: {theorem_name}

Proposed hypotheses:
{hypotheses_list}

Evaluate whether this list is necessary and sufficient.

Return ONLY the JSON.
"""


# ============================================================
# PARSER JSON
# ============================================================

def parse_llm_output(raw_text):

    if raw_text is None:
        return {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": "no_response",
        }

    cleaned = re.sub(
        r"```json|```",
        "",
        raw_text,
        flags=re.IGNORECASE,
    ).strip()

    try:

        parsed = json.loads(cleaned)

        return {
            "label": parsed.get("label"),
            "error_type": parsed.get("error_type"),
            "reason": parsed.get("reason"),
            "parse_error": None,
        }

    except Exception as e:

        return {
            "label": None,
            "error_type": None,
            "reason": None,
            "parse_error": str(e),
        }






def build_user_prompt(theorem_name, hypotheses):
    hyp = "\n".join(f"- {h}" for h in hypotheses)

    return USER_PROMPT_TEMPLATE.format(
        theorem_name=theorem_name,
        hypotheses_list=hyp,
    )



def call_llm(theorem_name, hypotheses):

    user_prompt = build_user_prompt(
        theorem_name,
        hypotheses,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )

    return answer.strip()

# df doit être ton DataFrame en mémoire (celui produit par le run précédent)
# Si tu repars d'un CSV sauvegardé, décommente la ligne suivante :
df = pd.read_csv("/kaggle/working/benchmark_results.csv")

MAX_RETRIES_PER_ROW = 3  # nombre de tentatives de relance par ligne avant abandon

failed_mask = df["parse_error"].notna()
failed_indices = df[failed_mask].index

print(f"{len(failed_indices)} lignes à ré-exécuter.")

for idx in failed_indices:
    row = df.loc[idx]
    theorem_name = row["name"]

    # "hypotheses" peut être une liste (si df encore en mémoire) ou une string (si relu depuis CSV)
    hyps = row["hypotheses"]
    if isinstance(hyps, str):
        import ast
        hyps = ast.literal_eval(hyps)

    print(f"\nRé-exécution : {row['theorem_id']} - {theorem_name}")

    success = False
    for attempt in range(1, MAX_RETRIES_PER_ROW + 1):
        try:
            raw = call_llm(theorem_name, hyps)
            parsed = parse_llm_output(raw)

            if parsed["parse_error"] is None:
                # Succès : on met à jour la ligne dans df
                df.loc[idx, "prediction_label"] = parsed["label"]
                df.loc[idx, "prediction_error"] = parsed["error_type"]
                df.loc[idx, "prediction_reason"] = parsed["reason"]
                df.loc[idx, "parse_error"] = None
                df.loc[idx, "raw_response"] = raw
                print(f"  -> Succès (tentative {attempt})")
                success = True
                break
            else:
                print(f"  -> Tentative {attempt} échouée (parse_error: {parsed['parse_error']})")

        except Exception as e:
            print(f"  -> Tentative {attempt} : erreur d'exécution ({e})")

        time.sleep(1)

    if not success:
        print(f"  -> ABANDON après {MAX_RETRIES_PER_ROW} tentatives pour {row['theorem_id']}")

# --- Sauvegarde du DataFrame mis à jour ---
OUTPUT_CSV_PATH = "/kaggle/working/benchmark_results.csv"
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nDataFrame mis à jour sauvegardé dans {OUTPUT_CSV_PATH}")

# --- Vérification finale ---
remaining_failed = df["parse_error"].notna().sum()
print(f"Lignes encore en échec après ré-exécution : {remaining_failed}")